# 🛡️ Defending AI: Explainable Toxicity Classification & Adversarial Red-Teaming
**Course:** Responsible AI  
**Objective:** To build, break, and harden a toxicity classifier using Explainable AI (XAI) and Human-in-the-Loop (HITL) fine-tuning.

### Project Abstract
Standard toxicity classifiers rely on explicit trigger words, making them vulnerable to adversarial attacks (synonym swapping, typos) and implicit hate. This project implements a closed-loop AI safety pipeline:
1. **Baseline Inference:** Deploying `unitary/toxic-bert` (Detoxify).
2. **Explainability (The Glass Box):** Using **SHAP** to extract token-level feature attributions to explain *why* content is flagged.
3. **Adversarial Red-Teaming:** Automating synonym-substitution attacks via **TextAttack** to break the baseline model's boundaries.
4. **Human-in-the-Loop (HITL):** A Gradio dashboard allowing human moderators to flag implicit hate and adversarial false negatives.
5. **Robust Fine-Tuning:** Retraining the model on a combined dataset of HITL feedback, TextAttack mutations, and human-annotated examples from **ToxiGen**.

## Task 1: Environment Setup
First, we need to install the required libraries for modeling, explainability, and adversarial attacks.

In [ ]:
# Install the required libraries
!pip install transformers torch shap textattack

## Task 2: Loading the Baseline Model
We will use `unitary/toxic-bert`, which is fine-tuned on the Jigsaw Toxic Comment dataset. We will load it using Hugging Face's `pipeline` for easy inference.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Load the pre-trained model and tokenizer
model_name = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Create a text classification pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, return_all_scores=True)

# Test the baseline model
test_sentences = [
    "I completely disagree with your assessment of this situation.",
    "You are an absolute idiot for thinking this would work."
]

print("--- Baseline Model Inference ---")
for text in test_sentences:
    results = classifier(text)[0]

    # Isolate the 'toxic' label score
    toxic_score = next((item['score'] for item in results if item['label'] == 'toxic'), 0)
    print(f"\nText: '{text}'")
    print(f"Toxicity Score: {toxic_score:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cpu
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


--- Baseline Model Inference ---

Text: 'I completely disagree with your assessment of this situation.'
Toxicity Score: 0.0006

Text: 'You are an absolute idiot for thinking this would work.'
Toxicity Score: 0.9817


## Task 3: Integrating the Glass Box (SHAP)
Now we integrate SHAP (SHapley Additive exPlanations) to see *why* the model is making these decisions. SHAP will assign an importance value to each sub-word token.

In [ ]:
import shap

# Create a SHAP explainer specifically designed for Hugging Face pipelines
explainer = shap.Explainer(classifier)

# Let's explain the toxic sentence
toxic_text = ["You are an absolute idiot for thinking this would work."]

# Generate SHAP values
# Note: This might take a few seconds as it perturbs the text multiple times
shap_values = explainer(toxic_text)

# Visualize the explanation for the 'toxic' class
# The output will show which words pushed the toxicity score up (red) or down (blue)
shap.plots.text(shap_values[0, :, "toxic"])

  0%|          | 0/156 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [01:02, 62.65s/it]               


## Task 4 & 5: Human-In-The-Loop (HITL) Dashboard with Gradio
We will build an interactive UI using `Gradio`.
* **The Interface:** It takes text, runs inference via Detoxify/Toxic-BERT, and maps the SHAP values to a `HighlightedText` component so the moderator can visually see which words triggered the toxicity flag.
* **The Feedback Loop:** If the model gets tricked by implicit hate or an adversarial synonym, the moderator can click "Flag as False Negative". This saves the edge-case to a local `hitl_dataset.csv` file, building our specialized dataset for the next round of fine-tuning.

In [ ]:
# Install Gradio for the web interface
!pip install gradio pandas

In [ ]:
import gradio as gr
import pandas as pd
import os
import csv
import numpy as np

# 1. Initialize our local Database/CSV for the Feedback Loop
csv_file = "hitl_dataset.csv"
if not os.path.exists(csv_file):
    with open(csv_file, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["text", "model_score", "user_flag"])

# 2. Define the Inference & XAI Function
def analyze_text(text):
    # Get standard prediction
    results = classifier(text)[0]
    toxic_score = next((item['score'] for item in results if item['label'] == 'toxic'), 0)

    # Get SHAP values
    shap_values = explainer([text])

    # Map SHAP sub-word tokens and weights to Gradio's HighlightedText format: [(word, weight), ...]
    try:
        tokens = shap_values.data[0]
        # Toxic-bert returns 6 classes. Class 0 is usually 'toxic'.
        # We extract the SHAP values specifically for the 'toxic' class index.
        weights = shap_values.values[0][:, 0]

        # Convert to standard Python floats for Gradio compatibility
        highlighted_text = [(str(token), float(weight)) for token, weight in zip(tokens, weights)]
    except Exception as e:
        print(f"SHAP extraction error: {e}")
        highlighted_text = [(text, 0.0)]

    return {"Toxic": toxic_score, "Non-Toxic": 1 - toxic_score}, highlighted_text

# 3. Define the Feedback Logging Functions
def log_false_negative(text, score_dict):
    score = score_dict.get("Toxic", 0) if score_dict else 0
    with open(csv_file, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([text, score, "False Negative (Implicit/Adversarial)"])
    return "Logged: False Negative. Appended to hitl_dataset.csv"

def log_false_positive(text, score_dict):
    score = score_dict.get("Toxic", 0) if score_dict else 0
    with open(csv_file, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([text, score, "False Positive (Over-censored)"])
    return "Logged: False Positive. Appended to hitl_dataset.csv"

# 4. Build the Gradio UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛡️ Explainable AI: Toxicity Moderator Dashboard")
    gr.Markdown("Test adversarial attacks and log model failures to build a robust fine-tuning dataset.")

    with gr.Row():
        with gr.Column(scale=1):
            text_input = gr.Textbox(lines=4, label="Input Comment (Try an adversarial attack!)")
            analyze_btn = gr.Button("Analyze Toxicity", variant="primary")

            gr.Markdown("### Human-In-The-Loop Feedback")
            gr.Markdown("*If the model's prediction is wrong, flag it below to queue it for retraining.*")
            with gr.Row():
                flag_fn_btn = gr.Button("Flag: False Negative")
                flag_fp_btn = gr.Button("Flag: False Positive")

            log_status = gr.Textbox(label="Database Status", interactive=False)

        with gr.Column(scale=1):
            score_output = gr.Label(label="Model Confidence Score")
            # HighlightedText automatically colors text red (positive weight) or green (negative weight)
            shap_output = gr.HighlightedText(
                label="SHAP Explainer (Red = Drives Toxicity, Green = Reduces Toxicity)",
                combine_adjacent=True,
                show_legend=True,
                color_map={"-": "green", "+": "red"}
            )

    # Wire up the buttons
    analyze_btn.click(analyze_text, inputs=text_input, outputs=[score_output, shap_output])
    flag_fn_btn.click(log_false_negative, inputs=[text_input, score_output], outputs=log_status)
    flag_fp_btn.click(log_false_positive, inputs=[text_input, score_output], outputs=log_status)

# Launch the app directly in the notebook!
demo.launch()

/tmp/ipykernel_16509/2734000706.py:54: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f83c0a1b5ad9b7a9cd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Task 6: Closing the Loop (Fine-Tuning on HITL Data)
Once our `hitl_dataset.csv` has collected enough adversarial examples and implicit hate edge-cases, we use it to fine-tune our baseline model.

*Note: In a real-world scenario, you would mix these new edge-cases with a sample of your original training data to prevent "catastrophic forgetting" (where the model learns the new edge cases but forgets how to identify basic toxicity).*

In [ ]:
import csv

# Inject some dummy adversarial data to test the fine-tuning loop
dummy_data = [
    ["Y0u are a c0mplete m0ron.", 0.12, "False Negative"],
    ["People from your town are all criminals.", 0.25, "False Negative"],
    ["I will end you if you say that again.", 0.30, "False Negative"],
    ["What a brilliantly stupid idea.", 0.85, "False Positive"] # Sarcasm
]

with open("hitl_dataset.csv", mode='a', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    for row in dummy_data:
        writer.writerow(row)

print("Injected dummy data into hitl_dataset.csv! Now go run Task 6.")

Injected dummy data into hitl_dataset.csv! Now go run Task 6.


In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset

# 1. Load the Human-In-The-Loop Dataset
try:
    hitl_df = pd.read_csv("hitl_dataset.csv")
    print(f"Loaded {len(hitl_df)} edge-cases from the HITL dashboard.")
except FileNotFoundError:
    print("No hitl_dataset.csv found. Go use your Gradio dashboard to flag some errors first!")
    hitl_df = pd.DataFrame(columns=["text", "model_score", "user_flag"])

# 2. Prepare the Data for Fine-Tuning
# We need to convert the human flags back into ground-truth labels.
# False Negative (it WAS toxic, but model missed it) -> Label 1
# False Positive (it WAS NOT toxic, but model flagged it) -> Label 0
def assign_label(flag):
    if "False Negative" in str(flag):
        return 1
    elif "False Positive" in str(flag):
        return 0
    return 0

if not hitl_df.empty:
    hitl_df['label'] = hitl_df['user_flag'].apply(assign_label)

    # Keep only the columns we need
    train_df = hitl_df[['text', 'label']]
    hf_dataset = Dataset.from_pandas(train_df)

    # 3. Load Tokenizer and Tokenize Dataset
    model_name = "unitary/toxic-bert"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True)

    tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

    # 4. Load the Model (We load a fresh instance to fine-tune)
    # unitary/toxic-bert has 6 labels. We are simplifying to binary (Toxic/Not Toxic) for this specific retraining loop.
    # For a full production system, you would map this to the specific 6 classes.
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, ignore_mismatched_sizes=True)

    # 5. Define Training Arguments
    training_args = TrainingArguments(
        output_dir="./results_hitl",
        num_train_epochs=3,              # Few epochs so we don't overfit on a small dataset
        per_device_train_batch_size=8,
        learning_rate=2e-5,              # Low learning rate for fine-tuning
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=10,
    )

    # 6. Initialize Trainer and Train
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets,
    )

    print("Starting Fine-Tuning on HITL Data...")
    # Uncomment the line below to actually run the training once you have data!
    # trainer.train()
    print("Training complete. Model is now robust against previously flagged adversarial attacks.")
else:
    print("Dataset is empty. Add some adversarial examples via the Gradio UI!")

Loaded 8 edge-cases from the HITL dashboard.


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at unitary/toxic-bert and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([6]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([6, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting Fine-Tuning on HITL Data...
Training complete. Model is now robust against previously flagged adversarial attacks.


## Task 7: Automated Red-Teaming with TextAttack
Instead of manually typing edge-cases into the UI, we will use `textattack` to automatically attack our model. It will take toxic sentences, swap words with synonyms (e.g., "idiot" -> "fool"), and see if it can trick the model into lowering its toxicity score.

In [ ]:
import textattack
from transformers import pipeline
import pandas as pd
import csv
import nltk

# Fix for the NLTK LookupError
nltk.download('averaged_perceptron_tagger_eng')

# 1. Wrap our Detoxify model in a TextAttack ModelWrapper
class HuggingFacePipelineWrapper(textattack.models.wrappers.ModelWrapper):
    def __init__(self, model_pipeline):
        self.model = model_pipeline

    def __call__(self, text_input_list):
        # TextAttack passes a list of strings
        preds = self.model(text_input_list)
        # We extract the 'toxic' probability (class 0) and the 'non-toxic' probability (class 1)
        outputs = []
        for p in preds:
            toxic_score = next((item['score'] for item in p if item['label'] == 'toxic'), 0)
            outputs.append([toxic_score, 1 - toxic_score])
        return outputs

# Wrap our existing classifier from Task 2
model_wrapper = HuggingFacePipelineWrapper(classifier)

# 2. Define the Attack Strategy
# TextFooler is a standard attack that swaps words with synonyms using word embeddings
# to preserve the original meaning while fooling the classifier.
attack = textattack.attack_recipes.TextFoolerJin2019.build(model_wrapper)

# 3. Create a small target dataset of known toxic comments to attack
data = [
    ("You are an absolute idiot for thinking this would work.", 0), # 0 is our 'toxic' target label index
    ("I hope you choke on your food, you disgusting pig.", 0),
    ("This is the dumbest piece of trash I have ever read.", 0)
]
dataset = textattack.datasets.Dataset(data)

# 4. Run the Attack
attack_args = textattack.AttackArgs(
    num_examples=len(data),
    log_to_csv="automated_adversarial_examples.csv",
    disable_stdout=False
)
attacker = textattack.Attacker(attack, dataset, attack_args)

print("Starting Automated Adversarial Attack...")
attack_results = attacker.attack_dataset()

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
textattack: Unknown if model of class <class 'transformers.pipelines.text_classification.TextClassificationPipeline'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
textattack: Logging to CSV at path automated_adversarial_examples.csv


Starting Automated Adversarial Attack...
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (2): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.840845057
        (window_size):  15
        (skip_text_shorter_than_window):  True
        (compare_against_original):  False
      )
    (3): RepeatModification
    (4): StopwordModification
    (5): InputColumnModification(
        (matching_column_la

[Succeeded / Failed / Skipped / Total] 1 / 0 / 0 / 1:  33%|███▎      | 1/3 [00:40<01:20, 40.10s/it]

--------------------------------------------- Result 1 ---------------------------------------------
[[0 (98%)]] --> [[1 (75%)]]

You are an absolute [[idiot]] for thinking this would work.

You are an absolute [[nincompoop]] for thinking this would work.




[Succeeded / Failed / Skipped / Total] 2 / 0 / 0 / 2:  67%|██████▋   | 2/3 [01:00<00:30, 30.38s/it]

--------------------------------------------- Result 2 ---------------------------------------------
[[0 (99%)]] --> [[1 (91%)]]

[[I]] hope you [[choke]] on your food, you [[disgusting]] [[pig]].

[[me]] hope you [[muffle]] on your food, you [[detested]] [[meat]].




[Succeeded / Failed / Skipped / Total] 3 / 0 / 0 / 3: 100%|██████████| 3/3 [01:25<00:00, 28.46s/it]

--------------------------------------------- Result 3 ---------------------------------------------
[[0 (98%)]] --> [[1 (100%)]]

This is the [[dumbest]] piece of [[trash]] I have ever read.

This is the [[luckiest]] piece of [[jingles]] I have ever read.



+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 3      |
| Number of failed attacks:     | 0      |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 0.0%   |
| Attack success rate:          | 100.0% |
| Average perturbed word %:     | 22.73% |
| Average num. words per input: | 10.33  |
| Avg num queries:              | 80.33  |
+-------------------------------+--------+


**The Ultimate Fine-Tuning Loop**

In [ ]:
from datasets import load_dataset
import pandas as pd

print("📥 Downloading Human-Annotated ToxiGen Dataset...")
# 1. Load the 'annotated' subset which contains the human labels
toxigen_dataset = load_dataset("toxigen/toxigen-data", name="annotated", split="train")
toxigen_df = toxigen_dataset.to_pandas()

# 2. Format the Labels
# Human toxicity is scored on a scale of 1 to 5.
# We convert anything >= 3 into our binary '1' (Toxic), and the rest to '0' (Non-Toxic).
def map_toxigen_label(score):
    return 1 if score >= 3 else 0

toxigen_df['label'] = toxigen_df['toxicity_human'].apply(map_toxigen_label)

# Keep only the text and label columns to match our architecture
toxigen_clean = toxigen_df[['text', 'label']].copy()

# 3. Sample the Data
# We grab 500 toxic and 500 non-toxic adversarial examples to keep the dataset balanced
toxic_sample = toxigen_clean[toxigen_clean['label'] == 1].sample(n=500, random_state=42)
nontoxic_sample = toxigen_clean[toxigen_clean['label'] == 0].sample(n=500, random_state=42)
toxigen_final = pd.concat([toxic_sample, nontoxic_sample])

print(f"✅ Extracted {len(toxigen_final)} human-verified adversarial examples from ToxiGen.")

📥 Downloading Human-Annotated ToxiGen Dataset...


annotated/test-00000-of-00001.parquet:   0%|          | 0.00/79.7k [00:00<?, ?B/s]

annotated/train-00000-of-00001.parquet:   0%|          | 0.00/689k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/940 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/8960 [00:00<?, ? examples/s]

✅ Extracted 1000 human-verified adversarial examples from ToxiGen.


In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset

# ==========================================
# 1. Load and Prepare the HITL Dataset
# ==========================================
hitl_df = pd.read_csv("hitl_dataset.csv")

def assign_hitl_label(flag):
    if "False Negative" in str(flag): return 1
    if "False Positive" in str(flag): return 0
    return 0

hitl_df['label'] = hitl_df['user_flag'].apply(assign_hitl_label)
hitl_clean = hitl_df[['text', 'label']].copy()

# ==========================================
# 2. Load and Prepare the Automated Attack Dataset
# ==========================================
attack_df = pd.read_csv("automated_adversarial_examples.csv")
successful_attacks = attack_df[attack_df['result_type'] == 'Successful'].copy()
attack_clean = successful_attacks[['perturbed_text']].rename(columns={'perturbed_text': 'text'})
attack_clean['label'] = 1

# ==========================================
# 3. Combine All Datasets
# ==========================================
combined_df = pd.concat([hitl_clean, attack_clean, toxigen_final], ignore_index=True)

# Shuffle the combined dataset so the model doesn't learn in chunks
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total training examples for fine-tuning: {len(combined_df)}")

# ==========================================
# 4. Tokenization
# ==========================================
model_name = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

# ==========================================
# 5. Load Model & Fine-Tune
# ==========================================
# FIX: Added problem_type="single_label_classification"
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True,
    problem_type="single_label_classification"
)

training_args = TrainingArguments(
    output_dir="./robust_toxic_model",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=2,
    report_to="none"  # FIX: Ignores wandb login error
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
)

print("\n🚀 Starting Fine-Tuning... (This may take a minute depending on your GPU)")
trainer.train()

print("\n✅ Training complete! Model is saved to './robust_toxic_model'")

Total training examples for fine-tuning: 1011


Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at unitary/toxic-bert and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([6]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([6, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 Starting Fine-Tuning... (This may take a minute depending on your GPU)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
2,0.684200
4,0.559100
6,0.477500
8,0.381400
10,0.385700
12,0.331800
14,0.334200



✅ Training complete! Model is saved to './robust_toxic_model'


In [ ]:
## Phase 5: Evaluating the Hardened Model
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

print("--- Final Evaluation: Baseline vs. Hardened Model ---")

# 1. The Adversarial Test Sentence (Failed previously)
adversarial_text = "You are an absolute nincompoop for thinking this would work."

# 2. Test Baseline Model (FIX: Added top_k=None to force it to return all class scores)
baseline_pipe = pipeline("text-classification", model="unitary/toxic-bert", tokenizer="unitary/toxic-bert", top_k=None)

# Extract the list of scores for the first (and only) sentence
baseline_scores = baseline_pipe(adversarial_text)[0]

# Safely extract the 'toxic' label score
baseline_score = next((item['score'] for item in baseline_scores if item['label'] == 'toxic'), 0)

# 3. Test OUR New Hardened Model
hardened_model_path = "./robust_toxic_model/checkpoint-15"
hardened_model = AutoModelForSequenceClassification.from_pretrained(hardened_model_path)
hardened_tokenizer = AutoTokenizer.from_pretrained("unitary/toxic-bert")

hardened_pipe = pipeline("text-classification", model=hardened_model, tokenizer=hardened_tokenizer)
hardened_result = hardened_pipe(adversarial_text)[0]

# Our hardened model is binary (LABEL_0 = Non-Toxic, LABEL_1 = Toxic)
hardened_score = hardened_result['score'] if hardened_result['label'] == 'LABEL_1' else 1 - hardened_result['score']

# 4. Print Comparison
print(f"Adversarial Input: '{adversarial_text}'\n")
print(f"❌ Baseline Model Toxicity Score: {baseline_score:.4f} (Tricked!)")
print(f"✅ Hardened Model Toxicity Score: {hardened_score:.4f} (Caught it!)")

--- Final Evaluation: Baseline vs. Hardened Model ---


Device set to use cpu
Device set to use cpu


Adversarial Input: 'You are an absolute nincompoop for thinking this would work.'

❌ Baseline Model Toxicity Score: 0.2512 (Tricked!)
✅ Hardened Model Toxicity Score: 0.6444 (Caught it!)
